# 01 — Create Target Table for Zerobus Ingest

Creates a Delta table with the **same schema** as Part 1's Bronze table.
This is where Zerobus will push events directly — no files, no message bus.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

ZEROBUS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_zerobus"

## Create the target table

Same fields as the structured streaming Bronze table — this makes the
comparison direct.

In [2]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {ZEROBUS_TABLE} (
        event_id        STRING,
        machine_id      STRING,
        casino_floor    STRING,
        game_type       STRING,
        player_id       STRING,
        bet_amount      DOUBLE,
        win_amount      DOUBLE,
        event_timestamp STRING
    )
""")

print(f"Table created: {ZEROBUS_TABLE}")
spark.sql(f"DESCRIBE TABLE {ZEROBUS_TABLE}").show(truncate=False)

Table created: alexander_booth.streaming_demo.slot_events_zerobus
+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|event_id       |string   |NULL   |
|machine_id     |string   |NULL   |
|casino_floor   |string   |NULL   |
|game_type      |string   |NULL   |
|player_id      |string   |NULL   |
|bet_amount     |double   |NULL   |
|win_amount     |double   |NULL   |
|event_timestamp|string   |NULL   |
+---------------+---------+-------+



## Grant permissions to the service principal

The Zerobus SDK authenticates as the service principal, so it needs
`MODIFY` and `SELECT` access on the target table.

Replace `<your-sp-client-id>` with the actual client ID from your `.env`.

In [3]:
SP_CLIENT_ID = os.getenv("ZEROBUS_CLIENT_ID", "<your-sp-client-id>")

grants = [
    f"GRANT USE CATALOG ON CATALOG {UC_CATALOG} TO `{SP_CLIENT_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA {UC_CATALOG}.{UC_SCHEMA} TO `{SP_CLIENT_ID}`",
    f"GRANT SELECT, MODIFY ON TABLE {ZEROBUS_TABLE} TO `{SP_CLIENT_ID}`",
]

for grant_sql in grants:
    print(f"Running: {grant_sql}")
    spark.sql(grant_sql)

print("\nPermissions granted.")

Running: GRANT USE CATALOG ON CATALOG alexander_booth TO `882bd6e2-60f9-42ef-a9dc-7bdde1f394d7`
Running: GRANT USE SCHEMA ON SCHEMA alexander_booth.streaming_demo TO `882bd6e2-60f9-42ef-a9dc-7bdde1f394d7`
Running: GRANT SELECT, MODIFY ON TABLE alexander_booth.streaming_demo.slot_events_zerobus TO `882bd6e2-60f9-42ef-a9dc-7bdde1f394d7`

Permissions granted.
